In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import prince

from importlib.metadata import version
from pathlib import Path
from matplotlib.transforms import offset_copy

import io
import contextlib
from adjustText import adjust_text

In [2]:
%run LittRuP__import_functions.ipynb

In [3]:
# chemins vers fichiers Data et Images

BASE_DIR = Path.cwd()

if BASE_DIR.name == "Notebooks":
    BASE_DIR = BASE_DIR.parent

DAT_DIR = BASE_DIR / "Data"
IMG_DIR = BASE_DIR / "Images"
DOC_DIR = BASE_DIR / "Docs"

In [4]:
# import matrice réduite pour AC

matrix_reduced_profile = pd.read_csv(DAT_DIR / "LittRu_AC_matrix_reduced_profile.csv", sep=',', header=0)

In [5]:
# auteurs en index

matrix_reduced_profile = matrix_reduced_profile.set_index("Author")

In [6]:
matrix_reduced_profile.head()

,Amour,Apprentissage,Argent,Bonheur,Fantastique,Femme,Guerre,Intelligentsia,Liberté,Mort,Nature,Noblesse,Occident,Patrie,Révolution,Société,Souffrance,Traditions,Ville
Author,,,,,,,,,,,,,,,,,,,
A.OSTROVSKI,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0
A.TOLSTOÏ,1,0,0,0,0,0,1,1,0,0,0,0,0,0,1,0,0,0,0
AKSAKOV,0,1,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,1,0
AVVAKUM,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0
AÏTMATOV,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0


In [7]:
matrix_reduced_profile.shape

(40, 19)

**Refaire l'AC avec tous les axes possibles**

**Pour mesurer la qualité de représentation d’un point sur un axe ou un plan, on risque de ne pas évaluer correctement ce que représentent les axes 1–2 par rapport à l’ensemble de l’espace si on ne calcule que 2 axes.**

**AC complète**

In [8]:
# ============================================================
# 1. Préparation de la matrice pour l'AC complète
# ============================================================

X_CA = matrix_reduced_profile.copy()

# sécurité : suppression d'éventuelles lignes ou colonnes nulles
X_CA = X_CA.loc[
    X_CA.sum(axis=1) > 0,
    X_CA.sum(axis=0) > 0
]

# nombre maximal d'axes factoriels
n_axes = min(X_CA.shape) - 1

print("Nombre d'axes calculés :", n_axes)

# ============================================================
# 2. Analyse des correspondances complète
# ============================================================

ca = prince.CA(
    n_components=n_axes,
    n_iter=10,
    copy=True,
    check_input=True,
    engine="sklearn",
    random_state=0
)

ca = ca.fit(X_CA)

# ============================================================
# 3. Coordonnées principales
# ============================================================

# coordonnées des auteurs
row_coords = ca.row_coordinates(X_CA)

# coordonnées des thèmes
col_coords = ca.column_coordinates(X_CA)


# noms explicites des axes
row_axis_names = [
    f"Axe {axis_number}"
    for axis_number in range(1, row_coords.shape[1] + 1)
]

col_axis_names = [
    f"Axe {axis_number}"
    for axis_number in range(1, col_coords.shape[1] + 1)
]

row_coords.columns = row_axis_names
col_coords.columns = col_axis_names

Nombre d'axes calculés : 18


**Cos2 des auteurs**

In [9]:
# ============================================================
# 4. Distances au centre des auteurs
# ============================================================

# d_i² = somme des coordonnées principales au carré
# sur tous les axes de l'AC complète

author_distances_squared = (
    row_coords
    .pow(2)
    .sum(axis=1)
)

author_distances_squared.name = "Distance au centre d2"
author_distances_squared.index.name = "Author"

# ============================================================
# 5. Cos² des auteurs sur tous les axes
# ============================================================

# remplacement préventif d'une éventuelle distance nulle
author_distances_squared_safe = (
    author_distances_squared
    .replace(0, np.nan)
)

# cos²(i, k) = coordonnée(i, k)² / d_i²
author_cos2_all = (
    row_coords
    .pow(2)
    .div(author_distances_squared_safe, axis=0)
)

author_cos2_all.index.name = "Author"

# ============================================================
# 6. Cos² des auteurs sur le plan 1-2
# ============================================================

author_cos2_plan12 = author_cos2_all[
    ["Axe 1", "Axe 2"]
].copy()

author_cos2_plan12.columns = [
    "Cos2 axe 1",
    "Cos2 axe 2"
]

# qualité de représentation globale sur le plan 1-2
author_cos2_plan12["Cos2 plan 1-2"] = (
    author_cos2_plan12["Cos2 axe 1"]
    + author_cos2_plan12["Cos2 axe 2"]
)

# ajout de la distance d_i² utilisée comme dénominateur
author_cos2_plan12.insert(
    0,
    "Distance au centre d2",
    author_distances_squared
)

# classement décroissant selon la qualité de représentation
author_cos2_plan12 = (
    author_cos2_plan12
    .sort_values(
        by="Cos2 plan 1-2",
        ascending=False
    )
)

author_cos2_plan12.index.name = "Author"

# display(author_cos2_plan12.round(4))

# ============================================================
# 7. Export des cos² des auteurs
# ============================================================

# ------------------------------------------------------------
# 7.1. Export Excel : plusieurs feuilles
# ------------------------------------------------------------

output_file_authors = (
    DAT_DIR
    / "LittRu_AC_cos2_auteurs.xlsx"
)

with pd.ExcelWriter(
    output_file_authors,
    engine="openpyxl"
) as writer:

    author_cos2_all.to_excel(
        writer,
        sheet_name="Cos2 tous axes"
    )

    author_cos2_plan12.to_excel(
        writer,
        sheet_name="Cos2 plan 1-2"
    )

    author_distances_squared.to_frame().to_excel(
        writer,
        sheet_name="Distances d2"
    )

print(
    "Fichier Excel enregistré :",
    output_file_authors
)

# ------------------------------------------------------------
# 7.2. Exports CSV : un fichier par tableau
# ------------------------------------------------------------

output_csv_author_cos2_all = (
    DAT_DIR
    / "LittRu_AC_cos2_auteurs_tous_axes.csv"
)

author_cos2_all.to_csv(
    output_csv_author_cos2_all,
    sep=",",
    encoding="utf-8-sig",
    index=True
)

output_csv_author_cos2_plan12 = (
    DAT_DIR
    / "LittRu_AC_cos2_auteurs_plan_1-2.csv"
)

author_cos2_plan12.to_csv(
    output_csv_author_cos2_plan12,
    sep=",",
    encoding="utf-8-sig",
    index=True
)

output_csv_author_distances_squared = (
    DAT_DIR
    / "LittRu_AC_distances_d2_auteurs.csv"
)

author_distances_squared.to_frame().to_csv(
    output_csv_author_distances_squared,
    sep=",",
    encoding="utf-8-sig",
    index=True
)

print("Fichiers CSV enregistrés :")
print(output_csv_author_cos2_all)
print(output_csv_author_cos2_plan12)
print(output_csv_author_distances_squared)


Fichier Excel enregistré : C:\Users\thaly\OneDrive\Documents\_Post-Bac-Python\Programs (LittRu - Publication 2026.07.15)\Data\LittRu_AC_cos2_auteurs.xlsx
Fichiers CSV enregistrés :
C:\Users\thaly\OneDrive\Documents\_Post-Bac-Python\Programs (LittRu - Publication 2026.07.15)\Data\LittRu_AC_cos2_auteurs_tous_axes.csv
C:\Users\thaly\OneDrive\Documents\_Post-Bac-Python\Programs (LittRu - Publication 2026.07.15)\Data\LittRu_AC_cos2_auteurs_plan_1-2.csv
C:\Users\thaly\OneDrive\Documents\_Post-Bac-Python\Programs (LittRu - Publication 2026.07.15)\Data\LittRu_AC_distances_d2_auteurs.csv


**Cos2 des thèmes**

In [10]:
# ============================================================
# 8. Distances au centre des thèmes
# ============================================================

# d_j² = somme des coordonnées principales au carré
# sur tous les axes de l'AC complète

theme_distances_squared = (
    col_coords
    .pow(2)
    .sum(axis=1)
)

theme_distances_squared.name = "Distance au centre d2"
theme_distances_squared.index.name = "Theme"

# ============================================================
# 9. Cos² des thèmes sur tous les axes
# ============================================================

# remplacement préventif d'une éventuelle distance nulle
theme_distances_squared_safe = (
    theme_distances_squared
    .replace(0, np.nan)
)

# cos²(j, k) = coordonnée(j, k)² / d_j²
theme_cos2_all = (
    col_coords
    .pow(2)
    .div(theme_distances_squared_safe, axis=0)
)

theme_cos2_all.index.name = "Theme"

# ============================================================
# 10. Cos² des thèmes sur le plan 1-2
# ============================================================

theme_cos2_plan12 = theme_cos2_all[
    ["Axe 1", "Axe 2"]
].copy()

theme_cos2_plan12.columns = [
    "Cos2 axe 1",
    "Cos2 axe 2"
]

# qualité de représentation globale sur le plan 1-2
theme_cos2_plan12["Cos2 plan 1-2"] = (
    theme_cos2_plan12["Cos2 axe 1"]
    + theme_cos2_plan12["Cos2 axe 2"]
)

# ajout de la distance d_j² utilisée comme dénominateur
theme_cos2_plan12.insert(
    0,
    "Distance au centre d2",
    theme_distances_squared
)

# classement décroissant selon la qualité de représentation
theme_cos2_plan12 = (
    theme_cos2_plan12
    .sort_values(
        by="Cos2 plan 1-2",
        ascending=False
    )
)

theme_cos2_plan12.index.name = "Theme"

# display(theme_cos2_plan12.round(4))

# ============================================================
# 11. Export des cos² des thèmes
# ============================================================

# ------------------------------------------------------------
# 11.1. Export Excel : plusieurs feuilles
# ------------------------------------------------------------

output_file_themes = (
    DAT_DIR
    / "LittRu_AC_cos2_themes.xlsx"
)

with pd.ExcelWriter(
    output_file_themes,
    engine="openpyxl"
) as writer:

    theme_cos2_all.to_excel(
        writer,
        sheet_name="Cos2 tous axes"
    )

    theme_cos2_plan12.to_excel(
        writer,
        sheet_name="Cos2 plan 1-2"
    )

    theme_distances_squared.to_frame().to_excel(
        writer,
        sheet_name="Distances d2"
    )

print(
    "Fichier Excel enregistré :",
    output_file_themes
)

# ------------------------------------------------------------
# 11.2. Exports CSV : un fichier par tableau
# ------------------------------------------------------------

output_csv_theme_cos2_all = (
    DAT_DIR
    / "LittRu_AC_cos2_themes_tous_axes.csv"
)

theme_cos2_all.to_csv(
    output_csv_theme_cos2_all,
    sep=",",
    encoding="utf-8-sig",
    index=True
)

output_csv_theme_cos2_plan12 = (
    DAT_DIR
    / "LittRu_AC_cos2_themes_plan_1-2.csv"
)

theme_cos2_plan12.to_csv(
    output_csv_theme_cos2_plan12,
    sep=",",
    encoding="utf-8-sig",
    index=True
)

output_csv_theme_distances_squared = (
    DAT_DIR
    / "LittRu_AC_distances_d2_themes.csv"
)

theme_distances_squared.to_frame().to_csv(
    output_csv_theme_distances_squared,
    sep=",",
    encoding="utf-8-sig",
    index=True
)

print("Fichiers CSV enregistrés :")
print(output_csv_theme_cos2_all)
print(output_csv_theme_cos2_plan12)
print(output_csv_theme_distances_squared)


Fichier Excel enregistré : C:\Users\thaly\OneDrive\Documents\_Post-Bac-Python\Programs (LittRu - Publication 2026.07.15)\Data\LittRu_AC_cos2_themes.xlsx
Fichiers CSV enregistrés :
C:\Users\thaly\OneDrive\Documents\_Post-Bac-Python\Programs (LittRu - Publication 2026.07.15)\Data\LittRu_AC_cos2_themes_tous_axes.csv
C:\Users\thaly\OneDrive\Documents\_Post-Bac-Python\Programs (LittRu - Publication 2026.07.15)\Data\LittRu_AC_cos2_themes_plan_1-2.csv
C:\Users\thaly\OneDrive\Documents\_Post-Bac-Python\Programs (LittRu - Publication 2026.07.15)\Data\LittRu_AC_distances_d2_themes.csv


**Vérifications**

In [11]:
# Chaque cos² est compris entre 0 et 1. 
# Pour un auteur donné, la somme des cos² sur tous les axes doit être égale à 1.

# vérification
author_cos2_sums = author_cos2_all.sum(
    axis=1,
    min_count=1
)

print(
    "Somme des cos2 égale à 1 pour tous les auteurs :",
    np.allclose(
        author_cos2_sums.dropna(),
        1.0
    )
)

Somme des cos2 égale à 1 pour tous les auteurs : True


In [12]:
# Chaque cos² est compris entre 0 et 1. 
# Pour un thème donné, la somme des cos² sur tous les axes doit être égale à 1.

theme_cos2_sums = theme_cos2_all.sum(
    axis=1,
    min_count=1
)

print(
    "Somme des cos2 égale à 1 pour tous les thèmes :",
    np.allclose(
        theme_cos2_sums.dropna(),
        1.0
    )
)

Somme des cos2 égale à 1 pour tous les thèmes : True
